In [1]:
# Notebook 3 — Embeddings prep, splits, and feature fusion
# This notebook:
# 1) Loads cleaned data (from Notebook 1)
# 2) Re-creates pattern transactions in the same way as Notebook 2 (for consistency)
# 3) Builds a binary sparse matrix from pattern items
# 4) Computes multilingual sentence embeddings for descriptions
# 5) Fuses pattern features + embeddings
# 6) Creates stratified train/val/test splits and saves artifacts

import os
import re
import json
import math
import random
import numpy as np
import pandas as pd
from datetime import datetime
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# IO paths
DATA_IN = "data/processed/descriptions_clean.csv"
RESULTS_DIR = "results"
PROCESSED_DIR = "data/processed"
REPORTS_DIR = "docs/reports"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# Mining configuration (keep consistent with Notebook 2)
MIN_SUP = 0.05
MAX_ITEMSET_LEN = 3
TOP_K_ITEMS_FALLBACK = 200

# Embedding configuration
# Use multilingual model to support English/Urdu/Roman Urdu
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
EMBED_BATCH_SIZE = 64

TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
print("Config ready. Paths OK. Seed set.")

Config ready. Paths OK. Seed set.


C:\Users\stran\AppData\Local\Temp\ipykernel_19396\184963049.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")


In [2]:
if not os.path.exists(DATA_IN):
    raise FileNotFoundError(f"{DATA_IN} not found. Run Notebook 1 first.")

df = pd.read_csv(DATA_IN)
print("Loaded:", DATA_IN, "| rows:", len(df))

# Expect key columns from Notebook 1
expected_cols = [
    "id","description_final","language","osm_tag_key","osm_tag_value","city","province","country"
]
missing_cols = [c for c in expected_cols if c not in df.columns]
if missing_cols:
    print("Warning: missing expected columns:", missing_cols)

# Target label as "key=value"
df["osm_label"] = df["osm_tag_key"].fillna("") + "=" + df["osm_tag_value"].fillna("")
print("Unique osm_label:", df["osm_label"].nunique())
df.head()

Loaded: data/processed/descriptions_clean.csv | rows: 3
Unique osm_label: 3


,id,osm_id,osm_geom_type,osm_tag_key,osm_tag_value,name,name_ur,alt_name,description_raw,description_final,...,wikipedia_title,wikipedia_url,lat,lon,city,province,country,language,dedup_group_id,osm_label
0,toy-1,n1,node,amenity,park,Kids Park F-7,بچوں کا پارک ایف-7,NaN,"A quiet park near the F-7 markaz, perfect for ...","A quiet park near the F-7 markaz, perfect for ...",...,NaN,NaN,33.723,73.055,Islamabad,ICT,Pakistan,Roman Urdu,0dc2b872-da697349,amenity=park
1,toy-2,w2,way,amenity,place_of_worship,Masjid-e-Quba,مسجد قبا,Quba Mosque,پرانے بازار کے قریب ایک خوبصورت مسجد۔,پرانے بازار کے قریب ایک خوبصورت مسجد۔,...,NaN,NaN,33.706,73.039,Islamabad,ICT,Pakistan,Urdu,05f94746-e0e4091e,amenity=place_of_worship
2,toy-3,n3,node,shop,mall,Centaurus,سینٹورس,The Centaurus Mall,Famous mall with food court and cinema.,Famous mall with food court and cinema.,...,The Centaurus,NaN,33.710,73.058,Islamabad,ICT,Pakistan,English,b6ece0a2-82d7bdcf,shop=mall


In [3]:
# We reproduce key helpers from Notebook 02_2_patternmining_cleaneddata_from_ASA.ipynb so this notebook is self-contained.
import nltk

# Ensure NLTK resources are available (safe to re-run)
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")
try:
    nltk.data.find("taggers/averaged_perceptron_tagger")
except LookupError:
    nltk.download("averaged_perceptron_tagger")

from nltk import word_tokenize, pos_tag

# Domain lexicons and cues (Pakistan/OSM-oriented)
OSM_VALUE_SYNONYMS = {
    "amenity=mosque": {"mosque", "masjid", "jamia", "imam-bargah", "imambargah"},
    "amenity=school": {"school", "madrasa", "college"},
    "amenity=hospital": {"hospital", "clinic"},
    "amenity=park": {"park", "family-park", "kid-park"},
    "shop=mall": {"mall", "markaz", "center", "centaurus"},
    "amenity=bazaar": {"bazaar", "bazar", "mandi", "market"},
    "highway=chowk": {"chowk", "roundabout"},
    "amenity=restaurant": {"restaurant", "hotel", "dhaba", "eatery"},
    "amenity=cafe": {"cafe", "coffee"},
    "amenity=bank": {"bank", "atm"},
    "amenity=university": {"university", "uni", "campus"},
}

ADJECTIVE_CUES = {
    "quiet","peaceful","beautiful","old","new","historic","famous","popular","big","small",
    "family","kids","cheap","expensive","crowded","busy","clean","green",
    "khubsurat","purana","naya","bari","choti","mashhoor"
}
NOUN_CUES = {
    "park","mosque","masjid","bazaar","bazar","market","mandi","chowk","roundabout","mall","markaz",
    "school","college","university","hospital","clinic","restaurant","cafe","bank",
    "lake","trail","museum","cinema","zoo"
}
URDU_NOUN_CUES = {
    "مسجد","بازار","چوک","پارک","اسکول","کالج","ہسپتال","کلیہ","ریسٹورنٹ","بینک","چڑیاگھر","سینما"
}

def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize(text: str, lang: str) -> list:
    text = clean_text(text)
    if not text:
        return []
    if lang in ("English", "Roman Urdu", "Mixed"):
        try:
            return word_tokenize(text)
        except Exception:
            return text.split()
    return text.split()

def pos_tag_safe(tokens: list, lang: str) -> list:
    if not tokens:
        return []
    if lang in ("English", "Roman Urdu", "Mixed"):
        try:
            return pos_tag(tokens)
        except Exception:
            return [(t, "X") for t in tokens]
    return [(t, "X") for t in tokens]

def extract_patterns(text: str, lang: str) -> set:
    # Returns a set of pattern items found in text
    patterns = set()
    toks = tokenize(text, lang)
    if not toks:
        return patterns

    if lang in ("English", "Mixed"):
        tagged = pos_tag_safe(toks, lang)
        # adj-noun
        for i in range(len(tagged) - 1):
            w1, p1 = tagged[i]
            w2, p2 = tagged[i+1]
            if p1.startswith("JJ") and p2.startswith("NN"):
                patterns.add(f"{w1}_{w2}")
        # noun-noun
        for i in range(len(tagged) - 1):
            w1, p1 = tagged[i]
            w2, p2 = tagged[i+1]
            if p1.startswith("NN") and p2.startswith("NN"):
                patterns.add(f"{w1}_{w2}")
        # cue singles
        for w, _ in tagged:
            if w in ADJECTIVE_CUES:
                patterns.add(f"adj:{w}")
            if w in NOUN_CUES:
                patterns.add(f"noun:{w}")

    elif lang == "Roman Urdu":
        # heuristic bigrams and singles
        for i in range(len(toks) - 1):
            w1, w2 = toks[i], toks[i+1]
            if w1 in ADJECTIVE_CUES and w2 in NOUN_CUES:
                patterns.add(f"{w1}_{w2}")
        for w in toks:
            if w in ADJECTIVE_CUES:
                patterns.add(f"adj:{w}")
            if w in NOUN_CUES:
                patterns.add(f"noun:{w}")

    elif lang == "Urdu":
        # minimal Urdu noun cues and previous token
        for i, w in enumerate(toks):
            if w in URDU_NOUN_CUES:
                patterns.add(f"noun_ur:{w}")
                if i > 0 and re.match(r"^\w+$", toks[i-1]):
                    patterns.add(f"ur_prev_{toks[i-1]}_{w}")

    else:
        for w in toks:
            if w in NOUN_CUES:
                patterns.add(f"noun:{w}")
    return patterns

In [4]:
transactions = []
labels = []
row_ids = []

for idx, row in df.iterrows():
    text = row.get("description_final", "") or ""
    lang = row.get("language", "Unknown")
    city = clean_text(row.get("city", "") or "")
    label = row.get("osm_label", "")
    rid = row.get("id", idx)

    items = set()
    # Extract patterns
    pats = extract_patterns(text, lang)
    items.update(pats)

    # City token (helps locality signals)
    if city:
        items.add(f"city:{city}")

    # Guided OSM synonyms
    text_clean = clean_text(text)
    for canonical, syns in OSM_VALUE_SYNONYMS.items():
        if any(s in text_clean for s in syns):
            items.add(f"osm_hint:{canonical}")

    transactions.append(items)
    labels.append(label)
    row_ids.append(rid)

print("Transactions:", len(transactions))
print("Non-empty transactions:", sum(1 for t in transactions if len(t) > 0))

Transactions: 3
Non-empty transactions: 3


In [5]:
pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Pruning strategy:
# - Keep all 'osm_hint:' items
# - Keep items with frequency >= ceil(MIN_SUP * N)
# - If still too many, keep top-K by frequency (hints always kept)

from scipy.sparse import csr_matrix

N = len(transactions)
freq = Counter()
for t in transactions:
    freq.update(t)

freq_min = max(2, math.ceil(MIN_SUP * N))
keep_items = set([it for it, c in freq.items() if c >= freq_min or it.startswith("osm_hint:")])

if len(keep_items) > TOP_K_ITEMS_FALLBACK:
    # Separate hints (always keep)
    hints = [it for it in keep_items if it.startswith("osm_hint:")]
    others = [(it, freq[it]) for it in keep_items if not it.startswith("osm_hint:")]
    others_sorted = sorted(others, key=lambda x: x[1], reverse=True)
    space = max(0, TOP_K_ITEMS_FALLBACK - len(hints))
    top_others = [it for it, _ in others_sorted[:space]]
    keep_items = set(hints + top_others)

# Build index for features
vocab_items = sorted(list(keep_items))
item_to_idx = {it: i for i, it in enumerate(vocab_items)}

# Create CSR binary matrix for pattern items: shape [N, V]
rows, cols, data = [], [], []
for i, t in enumerate(transactions):
    for it in t:
        j = item_to_idx.get(it)
        if j is not None:
            rows.append(i)
            cols.append(j)
            data.append(1)

X_patterns = csr_matrix((data, (rows, cols)), shape=(N, len(vocab_items)), dtype=np.float32)

print("Pattern matrix shape:", X_patterns.shape)
print("Avg non-zeros per row:", (X_patterns.nnz / max(1, N)))

Pattern matrix shape: (3, 3)
Avg non-zeros per row: 2.0


In [7]:
# Create label encoder mapping osm_label -> int
labels_series = pd.Series(labels)
unique_labels = sorted([lab for lab in labels_series.unique() if lab and lab != "="])
label_to_id = {lab: i for i, lab in enumerate(unique_labels)}
id_to_label = {i: lab for lab, i in label_to_id.items()}

# Map to ids; unknowns get -1 and will be filtered out from splits
y = labels_series.map(lambda lab: label_to_id.get(lab, -1)).values

# Filter rows with unknown labels
valid_mask = (y >= 0)
if valid_mask.sum() != len(y):
    print("Filtering out rows with unknown/empty label:", int((~valid_mask).sum()))
X_patterns = X_patterns[valid_mask]
df_valid = df.loc[valid_mask].reset_index(drop=True)
y = y[valid_mask]
print("After label filtering:", X_patterns.shape, "| classes:", len(unique_labels))

After label filtering: (3, 3) | classes: 3


In [3]:
pip install --upgrade pip setuptools wheel certifi urllib3 cryptography

  Using cached wheel-0.45.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached certifi-2025.8.3-py3-none-any.whl.metadata (2.4 kB)
  Using cached cryptography-45.0.6-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 5.7 MB/s eta 0:00:00
Using cached wheel-0.45.1-py3-none-any.whl (72 kB)
Using cached certifi-2025.8.3-py3-none-any.whl (161 kB)
Using cached cryptography-45.0.6-cp311-abi3-win_amd64.whl (3.4 MB)

   ---------------------------------------- 0/4 [wheel]
   ---------------------------------------- 0/4 [wheel]
  Attempting uninstall: pip
   ---------------------------------------- 0/4 [wheel]
   ---------- ----------------------------- 1/4 [pip]
    Found existing installation: pip 25.1.1
   ---------- ----------------------------- 1/4 [pip]
   ---------- ----------------------------- 1/4 

In [1]:
pip install sentence-transformers

  Using cached sentence_transformers-5.1.0-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-4.55.2-py3-none-any.whl.metadata (41 kB)
  Using cached torch-2.8.0-cp313-cp313-win_amd64.whl.metadata (30 kB)
  Using cached scikit_learn-1.7.1-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached huggingface_hub-0.34.4-py3-none-any.whl.metadata (14 kB)
  Using cached pillow-11.3.0-cp313-cp313-win_amd64.whl.metadata (9.2 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.6.2-cp38-abi3-win_amd64.whl.metadata (4.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached sentence_transformers-5.1.0-py3-none-any.whl (483 kB)
Using cached transformers-4.55.2-py3-none-any.whl (11.3 MB)
Using cached huggingface_hub

ERROR: Exception:
Traceback (most recent call last):
  File "E:\Github\repo-phaze7r\geospatial-tagging-thesis\.venv\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "E:\Github\repo-phaze7r\geospatial-tagging-thesis\.venv\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ~~~~~~~~~~~~~^^^^^
  File "E:\Github\repo-phaze7r\geospatial-tagging-thesis\.venv\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ~~~~~~~~~~~~~^^^^^
  File "E:\Github\repo-phaze7r\geospatial-tagging-thesis\.venv\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ~~~~~~~~~~~~~~^^^^^
  File "C:\Users\stran\miniconda3\Lib\http\client.py", line 479, in read
    s = self.fp.read(amt)
  File "C:\Users\stran\

In [1]:
# Compute sentence embeddings using sentence-transformers (multilingual)
# We use description_final as the primary text field.
try:
    from sentence_transformers import SentenceTransformer
    import torch
except Exception as e:
    raise RuntimeError(
        "sentence-transformers is required. Install with: pip install sentence-transformers\n"
        f"Import error: {e}"
    )

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
print("Embedding model loaded:", EMBED_MODEL_NAME, "| device:", device)

texts = df_valid["description_final"].fillna("").astype(str).tolist()

# Batch encode for efficiency
emb_list = []
for start in range(0, len(texts), EMBED_BATCH_SIZE):
    batch = texts[start:start+EMBED_BATCH_SIZE]
    embs = model.encode(batch, show_progress_bar=False, normalize_embeddings=True)
    emb_list.append(embs)
X_embed = np.vstack(emb_list).astype(np.float32)

print("Embeddings shape:", X_embed.shape)

RuntimeError: sentence-transformers is required. Install with: pip install sentence-transformers
Import error: No module named 'sentence_transformers'